In [0]:
%run ./00_config

In [0]:
from pyspark.sql.functions import col, count, when, isnan, countDistinct

def profile_dataset(name, path, format_opts={"header": "true", "inferSchema": "true"}):
    print(f"\n====================================================================")
    print(f"📊 PROFILING DATASET: {name}")
    print(f"====================================================================")
    try:
        # Load data using standard CSV reader
        df = spark.read.format("csv").options(**format_opts).load(path)
        
        # 1. Structural Review
        print(f"🔹 Total Rows: {df.count()}")
        print("\n🔹 Inferred Schema Structure:")
        df.printSchema()
        
        # 2. Defect Scan: Check for Null / Missing Values
        print("\n❌ DEFECT REPORT: Null/Missing Values Per Column:")
        null_counts = df.select([count(when(col(c).isNull() | isnan(col(c)) | (col(c) == ""), c)).alias(c) for c in df.columns]).collect()[0]
        for idx, col_name in enumerate(df.columns):
            print(f"  └─ {col_name}: {null_counts[idx]} null/missing records")
            
        # 3. Defect Scan: Check for Duplicate Keys
        # Guess primary key based on naming (e.g., customer_id, transaction_id)
        pk_candidates = [c for c in df.columns if "_id" in c]
        if pk_candidates:
            pk = pk_candidates[0]
            total_rows = df.count()
            unique_keys = df.select(pk).distinct().count()
            print(f"\n❌ DEFECT REPORT: Duplicate Key Verification ({pk}):")
            print(f"  └─ Total Rows: {total_rows} | Unique Keys: {unique_keys}")
            if total_rows > unique_keys:
                print(f"  ⚠️ ALERT: Duplicate values detected on Business Key '{pk}'!")
                
    except Exception as e:
        print(f"⚠️ Error profiling {name}: {str(e)}")

# Execute profile runs across all 4 uploaded source file groupings
profile_dataset("Customers", path_customers)
profile_dataset("Accounts", path_accounts)
profile_dataset("Branches", path_branches)
profile_dataset("Staging Batch 2 Transactions", f"{volume_root_path}/staging/transactions_batch_02.csv")
